In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Detect project root safely
cwd = Path.cwd()

if (cwd / "02_dataset").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "02_dataset").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate the project root.")

DATA_DIR = PROJECT_ROOT / "02_dataset" / "raw" / "CMAPSSData"

print("Project root:", PROJECT_ROOT)
print("Dataset directory:", DATA_DIR)
print("Dataset directory exists:", DATA_DIR.exists())

In [ ]:
print("Kernel is working")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

cwd = Path.cwd()

if (cwd / "02_dataset").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "02_dataset").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate the project root.")

DATA_DIR = PROJECT_ROOT / "02_dataset" / "raw" / "CMAPSSData"

print("Project root:", PROJECT_ROOT)
print("Dataset directory:", DATA_DIR)
print("Dataset directory exists:", DATA_DIR.exists())

In [ ]:
column_names = (
    ["unit_id", "cycle"]
    + [f"op_setting_{i}" for i in range(1, 4)]
    + [f"sensor_{i}" for i in range(1, 22)]
)

print("Number of expected columns:", len(column_names))
print(column_names)

In [ ]:
train_fd001 = pd.read_csv(
    DATA_DIR / "train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=column_names
)

test_fd001 = pd.read_csv(
    DATA_DIR / "test_FD001.txt",
    sep=r"\s+",
    header=None,
    names=column_names
)

rul_fd001 = pd.read_csv(
    DATA_DIR / "RUL_FD001.txt",
    sep=r"\s+",
    header=None,
    names=["RUL"]
)

print("FD001 train shape:", train_fd001.shape)
print("FD001 test shape :", test_fd001.shape)
print("FD001 RUL shape  :", rul_fd001.shape)

print("\nTrain engines:", train_fd001["unit_id"].nunique())
print("Test engines :", test_fd001["unit_id"].nunique())

In [ ]:
train_fd001 = pd.read_csv(
    DATA_DIR / "train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=column_names
)

test_fd001 = pd.read_csv(
    DATA_DIR / "test_FD001.txt",
    sep=r"\s+",
    header=None,
    names=column_names
)

rul_fd001 = pd.read_csv(
    DATA_DIR / "RUL_FD001.txt",
    sep=r"\s+",
    header=None,
    names=["RUL"]
)

print("FD001 train shape:", train_fd001.shape)
print("FD001 test shape :", test_fd001.shape)
print("FD001 RUL shape  :", rul_fd001.shape)

print("\nTrain engines:", train_fd001["unit_id"].nunique())
print("Test engines :", test_fd001["unit_id"].nunique())

In [ ]:
display(train_fd001.head())

In [ ]:
display(train_fd001.tail())

In [ ]:
train_fd001.groupby("unit_id")["cycle"].max().head(10)

In [ ]:
test_fd001.groupby("unit_id")["cycle"].max().head(10)

In [ ]:
display(rul_fd001.head(10))

In [ ]:
test_last_cycle = (
    test_fd001.groupby("unit_id")["cycle"]
    .max()
    .reset_index(name="last_observed_cycle")
)

test_rul_summary = rul_fd001.copy()
test_rul_summary["unit_id"] = np.arange(1, len(test_rul_summary) + 1)

test_summary = test_last_cycle.merge(
    test_rul_summary,
    on="unit_id"
)

test_summary["true_failure_cycle"] = (
    test_summary["last_observed_cycle"] + test_summary["RUL"]
)

display(test_summary.head(10))

In [ ]:
engine_1 = train_fd001[
    train_fd001["unit_id"] == 1
][["unit_id", "cycle"]].copy()

engine_1_max_cycle = engine_1["cycle"].max()

engine_1["RUL_raw"] = (
    engine_1_max_cycle - engine_1["cycle"]
)

display(engine_1.head())
display(engine_1.tail())

In [ ]:
print("Notebook is ready")

In [ ]:
print("Missing values in train:", train_fd001.isna().sum().sum())
print("Missing values in test :", test_fd001.isna().sum().sum())

print("Duplicate rows in train:", train_fd001.duplicated().sum())
print("Duplicate rows in test :", test_fd001.duplicated().sum())

In [ ]:
print("Train dtypes:")
print(train_fd001.dtypes.value_counts())

print("\nTest dtypes:")
print(test_fd001.dtypes.value_counts())

print("\nTraining cycle range:")
print(train_fd001["cycle"].min(), "to", train_fd001["cycle"].max())

print("\nTest cycle range:")
print(test_fd001["cycle"].min(), "to", test_fd001["cycle"].max())

In [ ]:
# Check duplicate (unit_id, cycle) combinations
train_duplicate_cycles = train_fd001.duplicated(
    subset=["unit_id", "cycle"]
).sum()

test_duplicate_cycles = test_fd001.duplicated(
    subset=["unit_id", "cycle"]
).sum()

print("Duplicate unit-cycle pairs in train:", train_duplicate_cycles)
print("Duplicate unit-cycle pairs in test :", test_duplicate_cycles)


# Check whether every engine starts at cycle 1
train_start_cycles = train_fd001.groupby("unit_id")["cycle"].min()
test_start_cycles = test_fd001.groupby("unit_id")["cycle"].min()

print("\nTrain engines not starting at cycle 1:",
      (train_start_cycles != 1).sum())

print("Test engines not starting at cycle 1 :",
      (test_start_cycles != 1).sum())


# Check whether cycle numbers are continuous for every engine
def check_cycle_continuity(df):
    problematic_engines = []

    for unit_id, group in df.groupby("unit_id"):
        cycles = group["cycle"].sort_values().to_numpy()

        expected_cycles = np.arange(
            cycles.min(),
            cycles.max() + 1
        )

        if not np.array_equal(cycles, expected_cycles):
            problematic_engines.append(unit_id)

    return problematic_engines


train_cycle_problems = check_cycle_continuity(train_fd001)
test_cycle_problems = check_cycle_continuity(test_fd001)

print("\nTrain engines with cycle gaps:", train_cycle_problems)
print("Test engines with cycle gaps :", test_cycle_problems)

In [ ]:
variability = pd.DataFrame({
    "n_unique": train_fd001.nunique(),
    "std": train_fd001.std(numeric_only=True)
}).sort_values(["n_unique", "std"])

display(variability)

In [ ]:
low_variability = variability[
    (variability["n_unique"] <= 10) |
    (variability["std"] < 1e-3)
]

display(low_variability)

In [ ]:
# Exact constant columns
constant_columns = [
    col for col in train_fd001.columns
    if train_fd001[col].nunique() == 1
]

print("Exact constant columns:")
print(constant_columns)


# Inspect selected low-variability columns more carefully
for col in ["sensor_6", "op_setting_1", "op_setting_2"]:
    print(f"\n--- {col} ---")
    print("Unique values:", train_fd001[col].nunique())
    print("Minimum:", train_fd001[col].min())
    print("Maximum:", train_fd001[col].max())
    print("\nMost common values:")
    print(train_fd001[col].value_counts().head(10))

In [ ]:
summary_stats = train_fd001.describe().T[
    ["mean", "std", "min", "max"]
]

display(summary_stats)

In [ ]:
sensor_columns = [f"sensor_{i}" for i in range(1, 22)]

sensor_summary = train_fd001[sensor_columns].describe().T[
    ["mean", "std", "min", "max"]
]

display(sensor_summary)